#### **Imported Libraries:**

In [1]:
import pygame
import random
from collections import deque


pygame 2.6.1 (SDL 2.28.4, Python 3.13.13)
Hello from the pygame community. https://www.pygame.org/contribute.html


##### **Game Sprits:**

In [2]:
def load_images():
    images = {}
    images["green"] = pygame.image.load("assets/green_grass.png")
    images["light"] = pygame.image.load("assets/light_grass.png")
    images["white_chicken"] = pygame.image.load("assets/white_chicken.png")
    images["black_chicken"] = pygame.image.load("assets/black_chicken.png")
    images["white_egg"] = pygame.image.load("assets/white_egg.png")
    images["black_egg"] = pygame.image.load("assets/dark_egg.png")
    images["trap"] = pygame.image.load("assets/trap_hole.webp")
    images["turd"] = pygame.image.load("assets/chick_turd.webp")

    return images

##### **Game Screen Constants:**

In [3]:
WIDTH = 800
HEIGHT = 600

ROWS = 8
COLS = 8

TILE_SIZE = 64

In [4]:
MOVES = {
    pygame.K_UP: (-1, 0),
    pygame.K_DOWN: (1, 0),
    pygame.K_LEFT: (0, -1),
    pygame.K_RIGHT: (0, 1)
}

##### **Game Board Drawing Utility:**

In [5]:
def draw_board(screen, images):
    for row in range(ROWS):
        for col in range(COLS):
            x = col * TILE_SIZE
            y = row * TILE_SIZE

            if (row + col) % 2 == 0:
                tile = images["green"]
            else:
                tile = images["light"]

            tile = pygame.transform.scale(tile, (TILE_SIZE, TILE_SIZE))
            screen.blit(tile, (x, y))

In [6]:
def reachable_tiles(start, traps):
    visited = set()
    queue = deque([start])

    while queue:
        r, c = queue.popleft()

        if (r, c) in visited:
            continue
        if (r, c) in traps:
            continue

        visited.add((r, c))

        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < ROWS and 0 <= nc < COLS:
                queue.append((nr, nc))

    return visited

In [7]:
def generate_trapdoors(num_traps=12):
    import random

    forbidden = {
        (0, 0), (0, 1), (1, 0),
        (7, 7), (7, 6), (6, 7)
    }

    while True:
        traps = set()

        while len(traps) < num_traps:
            r = random.randint(0, ROWS - 1)
            c = random.randint(0, COLS - 1)

            if (r, c) not in forbidden:
                traps.add((r, c))

        white_reach = reachable_tiles((0, 0), traps)
        black_reach = reachable_tiles((7, 7), traps)

        MIN_ACCESSIBLE = 15  # tweak if needed

        if len(white_reach) >= MIN_ACCESSIBLE and len(black_reach) >= MIN_ACCESSIBLE:
            return traps

In [8]:
def draw_traps(screen, state, images):
    for (r, c) in state.revealed_traps:
        x = c * TILE_SIZE
        y = r * TILE_SIZE

        img = pygame.transform.scale(images["trap"], (TILE_SIZE, TILE_SIZE))
        screen.blit(img, (x, y))

In [9]:
def draw_eggs(screen, state, images):
    for (r, c), player_id in state.eggs.items():
        x = c * TILE_SIZE
        y = r * TILE_SIZE

        img_key = "white_egg" if player_id == 0 else "black_egg"
        img = pygame.transform.scale(images[img_key], (TILE_SIZE, TILE_SIZE))
        screen.blit(img, (x, y))

In [10]:
def draw_turds(screen, state, images):
    for (r, c) in state.turds:
        x = c * TILE_SIZE
        y = r * TILE_SIZE

        img = pygame.transform.scale(images["turd"], (TILE_SIZE, TILE_SIZE))
        screen.blit(img, (x, y))

In [11]:
def draw_ui(screen, player1, player2, font, state):
    x_offset = COLS * TILE_SIZE + 20

    texts = [
        f"White Eggs: {player1.eggs_placed}",
        f"White Moves: {player1.moves_left}",
        f"White Turds: {player1.turds_remaining}",
        "",
        f"Black Eggs: {player2.eggs_placed}",
        f"Black Moves: {player2.moves_left}",
        f"Black Turds: {player2.turds_remaining}",
        "",
    ]

    current = get_current_player(state)
    turn_text = "White Turn" if current == player1 else "Black Turn"
    texts.append(turn_text)

    y = 50
    for text in texts:
        label = font.render(text, True, (0, 0, 0))
        screen.blit(label, (x_offset, y))
        y += 30

##### **Player Class and Functions:**

In [12]:
class Player:
    def __init__(self, row, col, image):
        self.row = row
        self.col = col
        self.image = image
        self.prev_row = row
        self.prev_col = col

        self.moves_left = 30
        self.turds_remaining = 4
        self.eggs_placed = 0

In [13]:
def draw_players(screen, players):
    for player in players:
        x = player.col * TILE_SIZE
        y = player.row * TILE_SIZE

        img = pygame.transform.scale(player.image, (TILE_SIZE, TILE_SIZE))
        screen.blit(img, (x, y))

In [14]:
def update_previous_position(player):
    player.prev_row = player.row
    player.prev_col = player.col

In [15]:
def get_current_player(state):
    return state.players[state.current_player_index]

In [16]:
def switch_turn(state):
    state.current_player_index = 1 - state.current_player_index

In [17]:
def move_player(state, direction):
    player = get_current_player(state)

    if player.moves_left <= 0:
        return "NO_MOVES"

    dr, dc = direction
    new_r = player.row + dr
    new_c = player.col + dc

    if not is_valid_move(state, player, new_r, new_c):
        return "ILLEGAL"

    # save previous
    update_previous_position(player)

    # move
    player.row = new_r
    player.col = new_c

    # resolve tile
    result = resolve_tile(state, player)

    if result == "TRAP":
        switch_turn(state)
        return "TRAP"

    return "SAFE"

In [18]:
def place_egg(state, player):
    pos = (player.row, player.col)

    if pos in state.eggs:
        return False
    if pos in state.turds:
        return False
    if pos in state.revealed_traps:
        return False

    state.eggs[pos] = state.current_player_index
    player.eggs_placed += 1

    return True

In [19]:
def place_turd(state, player):
    pos = (player.row, player.col)

    if player.turds_remaining <= 0:
        return False
    if pos in state.eggs or pos in state.turds:
        return False

    state.turds.add(pos)
    player.turds_remaining -= 1

    return True

##### **Expectiminimax Algorithm:**

In [20]:
# Make the agent value eggs more early-game and turds more mid/late-game.
def game_progress(snap):
    total = 60  # 30 + 30 moves initially
    remaining = snap.moves_left[0] + snap.moves_left[1]
    return 1 - (remaining / total)  # 0 early → 1 late

In [21]:
class StateSnapshot:
    """
    A lightweight, pygame-free copy of GameState used by the search.
    All positions are (row, col) tuples.
    """
    def __init__(self, state):
        p0, p1 = state.players[0], state.players[1]
        self.pos          = [(p0.row, p0.col), (p1.row, p1.col)]
        self.moves_left   = [p0.moves_left, p1.moves_left]
        self.eggs_placed  = [p0.eggs_placed, p1.eggs_placed]
        self.turds_rem    = [p0.turds_remaining, p1.turds_remaining]
        self.eggs         = dict(state.eggs)          # {(r,c): player_id}
        self.turds        = set(state.turds)
        self.traps        = set(state.traps)          # all traps (hidden + revealed)
        self.revealed_traps = set(state.revealed_traps)
        self.current      = state.current_player_index

    def copy(self):
        s = StateSnapshot.__new__(StateSnapshot)
        s.pos           = list(self.pos)
        s.moves_left    = list(self.moves_left)
        s.eggs_placed   = list(self.eggs_placed)
        s.turds_rem     = list(self.turds_rem)
        s.eggs          = dict(self.eggs)
        s.turds         = set(self.turds)
        s.traps         = set(self.traps)
        s.revealed_traps = set(self.revealed_traps)
        s.current       = self.current
        return s

In [22]:
# Weights for H(s) = W1*(E_ai − E_human) + W2*(A_ai − A_human) + W3*(Tu_ai − Tu_human)
W1 = 10   # egg score difference (most important)
W2 = 1    # available moves difference
W3 = 2    # remaining turds difference (turds = board control)

AGENT_IDX = 1   # black chicken is player index 1
HUMAN_IDX = 0

def trap_probability(snap):
    """
    P_trap = (12 - T_found) / U
    where T_found = number of revealed traps, U = number of unknown (unrevealed) tiles.
    """
    total_tiles = ROWS * COLS
    revealed_count = len(snap.revealed_traps) + len(snap.turds) + len(snap.eggs)
    unknown = total_tiles - revealed_count
    t_found = len(snap.revealed_traps)
    remaining_traps = 12 - t_found
    if unknown <= 0 or remaining_traps <= 0:
        return 0.0
    return min(remaining_traps / unknown, 1.0)

def heuristic(snap):
    ai, hu = AGENT_IDX, HUMAN_IDX

    progress = game_progress(snap)

    # --- Egg importance increases early ---
    egg_weight = 12 + (1 - progress) * 9   # early ≈ 20, late ≈ 12

    # --- Turd importance increases later ---
    turd_weight = 2 + progress * 8        # early ≈ 2, late ≈ 12

    e_diff = snap.eggs_placed[ai] - snap.eggs_placed[hu]
    a_diff = snap.moves_left[ai] - snap.moves_left[hu]

    # Turd pressure near human
    turd_score = 0
    hr, hc = snap.pos[hu]
    for (tr, tc) in snap.turds:
        dist = abs(hr - tr) + abs(hc - tc)
        if dist <= 2:
            turd_score += (3 - dist) * turd_weight

    return egg_weight * e_diff + 1 * a_diff + turd_score

In [23]:
DIRS = [(-1,0),(1,0),(0,-1),(0,1)]

def get_legal_actions(snap, player_idx):
    """
    Returns a list of actions the agent can take from its current tile.
    Each action: ('egg', target_pos) or ('turd', target_pos)

    The agent walks from its current position (no BFS/DFS),
    checking adjacent tiles until it finds a safe empty one.
    Trap probability is handled at Chance nodes, not here.
    """
    r, c   = snap.pos[player_idx]
    opp    = snap.pos[1 - player_idx]
    actions = []

    # Tiles the agent can reach by stepping one tile in each direction
    candidates = set()
    queue = [(r, c)]
    visited = {(r, c)}

    # Simple flood: find all reachable tiles (no BFS depth limit,
    # but skip known-blocked tiles only)
    from collections import deque
    q = deque([(r, c)])
    while q:
        cr, cc = q.popleft()
        for dr, dc in DIRS:
            nr, nc = cr + dr, cc + dc
            if not (0 <= nr < ROWS and 0 <= nc < COLS): continue
            if (nr, nc) in visited: continue
            if (nr, nc) == opp: continue
            if (nr, nc) in snap.turds: continue
            if (nr, nc) in snap.revealed_traps: continue
            visited.add((nr, nc))
            candidates.add((nr, nc))
            q.append((nr, nc))

    for pos in candidates:
        # Egg: tile must be empty (no egg, no turd, not a revealed trap)
        if pos not in snap.eggs and pos not in snap.turds and pos not in snap.revealed_traps:
            actions.append(('egg', pos))
        # Turd: tile must be empty, player must have turds left
        if snap.turds_rem[player_idx] > 0 and pos not in snap.eggs and pos not in snap.turds:
            actions.append(('turd', pos))

    return actions

In [24]:
def apply_action(snap, player_idx, action):
    """Returns a new StateSnapshot after applying action. Does NOT modify snap."""
    s = snap.copy()
    kind, pos = action

    if kind == 'egg':
        s.eggs[pos] = player_idx
        s.eggs_placed[player_idx] += 1
        s.moves_left[player_idx]  -= 1

    elif kind == 'turd':
        s.turds.add(pos)
        s.turds_rem[player_idx]  -= 1
        s.moves_left[player_idx] -= 1

    s.current = 1 - player_idx
    return s

In [25]:
SEARCH_DEPTH = 3   # keep low — algorithm is expensive

def expectiminimax(snap, depth, node_type):
    """
    node_type: 'max' | 'min' | 'chance'
    Returns a numeric score from the agent's perspective.
    """
    # Terminal / depth limit
    if depth == 0 or snap.moves_left[AGENT_IDX] <= 0 or snap.moves_left[HUMAN_IDX] <= 0:
        return heuristic(snap)

    if node_type == 'max':
        actions = get_legal_actions(snap, AGENT_IDX)
        if not actions:
            return heuristic(snap)
        best = float('-inf')
        for action in actions:
            child = apply_action(snap, AGENT_IDX, action)
            # After agent acts → chance node (will the human step on a trap?)
            val = expectiminimax(child, depth - 1, 'chance')
            if val > best:
                best = val
        return best

    elif node_type == 'min':
        actions = get_legal_actions(snap, HUMAN_IDX)
        if not actions:
            return heuristic(snap)
        worst = float('inf')
        for action in actions:
            child = apply_action(snap, HUMAN_IDX, action)
            val = expectiminimax(child, depth - 1, 'max')
            if val < worst:
                worst = val
        return worst

    else:  # chance
        p_trap = trap_probability(snap)
        p_safe = 1.0 - p_trap

        # --- Trap outcome ---
        trap_snap = snap.copy()
        trap_snap.moves_left[trap_snap.current] -= 2
        trap_snap.current = 1 - trap_snap.current   # ✅ turn ends

        trap_val = heuristic(trap_snap)

        # --- Safe outcome ---
        safe_snap = snap.copy()
        safe_snap.current = 1 - safe_snap.current   # after action, opponent plays
        safe_val = expectiminimax(safe_snap, depth - 1, 'min')

        return p_trap * trap_val + p_safe * safe_val

In [26]:
def agent_turn(state):
    """
    Called once per agent turn.
    """
    snap = StateSnapshot(state)
    actions = get_legal_actions(snap, AGENT_IDX)

    if not actions:
        # No moves available — force turn switch
        state.players[AGENT_IDX].moves_left -= 1
        switch_turn(state)
        return

    # Score each action with expectiminimax
    best_action = None
    best_score  = float('-inf')
    for action in actions:
        child = apply_action(snap, AGENT_IDX, action)

        # 🔥 NEW: penalize far targets
        (tr, tc) = action[1]
        ar, ac = snap.pos[AGENT_IDX]
        distance = abs(tr - ar) + abs(tc - ac)

        distance_penalty = -2 * distance   # tune this (1.0 → 2.5)

        score = expectiminimax(child, SEARCH_DEPTH - 1, 'chance') + distance_penalty

        if score > best_score:
            best_score = score
            best_action = action

    kind, (tr, tc) = best_action
    agent  = state.players[AGENT_IDX]

    # --- Walk the agent to the target tile, one step at a time ---
    max_steps = ROWS + COLS   
    steps = 0
    
    # FIX 1: Keep track of visited tiles during the walk to prevent infinite bouncing
    visited_walk = {(agent.row, agent.col)}

    while (agent.row, agent.col) != (tr, tc) and steps < max_steps:
        moved = False
        dr = tr - agent.row
        dc = tc - agent.col
        
        preferred = []
        if dr != 0: preferred.append((1 if dr > 0 else -1, 0))
        if dc != 0: preferred.append((0, 1 if dc > 0 else -1))

        # Build fallback directions safely
        fallback = [d for d in DIRS if d not in preferred]

        for ddr, ddc in preferred + fallback:
            nr, nc = agent.row + ddr, agent.col + ddc
            
            # Now it checks visited_walk so it is forced to move THROUGH the egg to an empty tile
            if is_valid_move(state, agent, nr, nc) and (nr, nc) not in visited_walk:
                update_previous_position(agent)
                agent.row, agent.col = nr, nc
                visited_walk.add((nr, nc)) # Log the step
                
                result = resolve_tile(state, agent)
                if result == "TRAP":
                    switch_turn(state)
                    return
                moved = True
                break

        if not moved:
            break  # completely stuck, give up

        steps += 1

    # --- Arrived at (or near) target — check tile is still usable ---
    pos = (agent.row, agent.col)

    if kind == 'egg' and pos not in state.eggs and pos not in state.turds and pos not in state.revealed_traps:
        place_egg(state, agent)
    elif kind == 'turd' and agent.turds_remaining > 0 and pos not in state.eggs and pos not in state.turds:
        place_turd(state, agent)
    else:
        # Target tile became occupied — move to any safe adjacent tile and try again
        for ddr, ddc in DIRS:
            nr, nc = agent.row + ddr, agent.col + ddc
            if is_valid_move(state, agent, nr, nc) and (nr, nc) not in state.eggs and (nr, nc) not in state.turds:
                update_previous_position(agent)
                agent.row, agent.col = nr, nc
                result = resolve_tile(state, agent)
                if result == "TRAP":
                    switch_turn(state)
                    return
                if kind == 'egg':
                    place_egg(state, agent)
                else:
                    place_turd(state, agent)
                break

    agent.moves_left -= 1
    switch_turn(state)

##### **Game Logic Utility:**

In [27]:
def handle_action_phase(state, player, event):
    if event.key == pygame.K_e:
        success = place_egg(state, player)

    elif event.key == pygame.K_t:
        success = place_turd(state, player)

    elif event.key == pygame.K_SPACE:
        success = True

    else:
        return  # wait for valid action key

    if success:
        player.moves_left -= 1
        switch_turn(state)

In [28]:
def is_valid_move(state, player, new_r, new_c):
    # outside board
    if not (0 <= new_r < ROWS and 0 <= new_c < COLS):
        return False

    # opponent position
    opponent = state.players[1 - state.current_player_index]
    if (new_r, new_c) == (opponent.row, opponent.col):
        return False

    # turd
    if (new_r, new_c) in state.turds:
        return False

    # revealed trap
    if (new_r, new_c) in state.revealed_traps:
        return False

    return True

In [29]:
def handle_input(state, event):
    # Ignore input during agent turn
    if state.current_player_index == AGENT_IDX:
        return

    if event.key in MOVES:
        result = move_player(state, MOVES[event.key])

        if result == "ILLEGAL":
            return

        if result == "TRAP":
            state.awaiting_action = False   # ✅ FIX: clear action phase
            return

        # SAFE → go to action phase
        state.awaiting_action = True
        return

    # Action phase
    if state.awaiting_action:
        player = get_current_player(state)
        handle_action_phase(state, player, event)
        state.awaiting_action = False

In [30]:
def resolve_tile(state, player):
    pos = (player.row, player.col)

    # Trapdoor
    if pos in state.traps:
        state.revealed_traps.add(pos)

        # penalty
        player.moves_left -= 2

        # return back
        player.row = player.prev_row
        player.col = player.prev_col

        return "TRAP"

    return "SAFE"

In [31]:
class GameState:
    def __init__(self, player1, player2, traps):
        self.players = [player1, player2]
        self.current_player_index = 0

        self.traps = traps
        self.revealed_traps = set()

        self.turds = set()
        self.eggs = {}  # (row, col) → player_id

        self.awaiting_action = False

In [32]:
pygame.init()

screen = pygame.display.set_mode((1000, 600))
pygame.display.set_caption("Eggplosion")

images = load_images()
font = pygame.font.SysFont(None, 24)

player1 = Player(0, 0, images["white_chicken"])
player2 = Player(7, 7, images["black_chicken"])

traps = generate_trapdoors()
state = GameState(player1, player2, traps)

players = [player1, player2]

running = True
clock = pygame.time.Clock()

while running:
    clock.tick(60)

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        if event.type == pygame.KEYDOWN:
            handle_input(state, event)

    # Agent acts automatically on its turn
    if state.current_player_index == AGENT_IDX and not state.awaiting_action:
        agent_turn(state)

    screen.fill((255, 255, 255))
    draw_board(screen, images)
    draw_traps(screen, state, images)
    draw_eggs(screen, state, images)
    draw_turds(screen, state, images)
    draw_players(screen, state.players)
    draw_ui(screen, player1, player2, font, state)
    pygame.display.flip()


: 